# 03 — Detecção de regimes via HMM 

Especificação do pré-projeto (seção 3.3): HMM gaussiano (covariância cheia)
sobre observações multivariadas — **retorno agregado da carteira +
volatilidade realizada** —, 2-4 estados via BIC/AIC + inspeção econômica,
Baum-Welch com múltiplas inicializações, Viterbi (caracterização in-sample),
Forward (inferência prospectiva, sem lookahead), robustez via bootstrap em
blocos.

**Este é o fit único/exploratório da Semana 4** (in-sample, para
caracterizar regimes) — o refit causal walk-forward de verdade, que gera a
feature que efetivamente alimenta o LSTM sem vazar informação futura para
os parâmetros do modelo, é escopo da Semana 6.

Duas especificações de observação, ambas do retorno agregado equal-weighted
(já convertido para BRL — ver notebook 02):
- **Principal (3D)**: `return_1d`, `realized_vol_21d`, `realized_vol_63d`
- **Robustez/parcimoniosa (2D)**: `return_1d`, `realized_vol_21d`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.hmm import (
    block_bootstrap_stability,
    characterize_regimes,
    fit_gaussian_hmm,
    forward_filter,
    select_n_states,
    viterbi_decode,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

portfolio_full = pd.read_parquet(PROCESSED_DIR / "portfolio_return.parquet")
valid = portfolio_full.notna().all(axis=1)
groups = valid.ne(valid.shift()).cumsum()
lengths = valid[valid].groupby(groups[valid]).size().tolist()
portfolio = portfolio_full.loc[valid]
print("Sequ?ncias cont?guas:", lengths)
portfolio.tail()

Sequ?ncias cont?guas: [207, 3989]


,return_1d,realized_vol_21d,realized_vol_63d
2024-12-20,-0.002995,0.180696,0.143490
2024-12-23,-0.016488,0.180925,0.146863
2024-12-26,0.006471,0.181341,0.146824
2024-12-27,-0.011832,0.188284,0.148794
2024-12-30,0.003306,0.185349,0.148803


## Especificação principal (3D): retorno + vol. realizada 21d + 63d

In [ ]:
cols_3d = ["return_1d", "realized_vol_21d", "realized_vol_63d"]
obs_3d = portfolio[cols_3d].values

selection_3d = select_n_states(obs_3d, state_range=(2, 3, 4), n_init=20, random_state=42, lengths=lengths)
selection_3d

In [ ]:
n_states_3d = selection_3d["bic"].idxmin()
print(f"BIC seleciona {n_states_3d} estados (AIC sugere {selection_3d['aic'].idxmin()})")

model_3d = fit_gaussian_hmm(obs_3d, n_states=n_states_3d, n_init=30, random_state=42, lengths=lengths)
regime_summary_3d = characterize_regimes(model_3d, obs_3d, cols_3d)
regime_summary_3d

## Viterbi (in-sample) vs. Forward (prospectivo, sem lookahead)

In [ ]:
viterbi_states = viterbi_decode(model_3d, obs_3d, lengths=lengths)
forward_probs = forward_filter(model_3d, obs_3d, lengths=lengths)
forward_state = forward_probs.argmax(axis=1)

regimes = pd.DataFrame(
    {
        "viterbi_state": viterbi_states,
        "forward_state": forward_state,
        **{f"forward_prob_state_{s}": forward_probs[:, s] for s in range(n_states_3d)},
    },
    index=portfolio.index,
)

agreement = (regimes["viterbi_state"] == regimes["forward_state"]).mean()
print(f"Viterbi (in-sample) concorda com Forward (prospectivo) em {agreement:.1%} dos dias")
print("(divergências esperadas nas transições de regime — Viterbi vê o futuro próximo, Forward não)")

regimes.to_parquet(PROCESSED_DIR / "hmm_regimes_insample.parquet")
regimes.tail()

## Robustez: bootstrap em blocos

Avalia estabilidade do nº de estados escolhido por BIC e da sequência de
regimes (Viterbi), com estados alinhados por volatilidade entre reamostragens
e comparados via % de concordância + Adjusted Rand Index.

In [ ]:
stability_3d = block_bootstrap_stability(
    obs_3d, state_range=(2, 3, 4), block_length=21, n_bootstrap=50, n_init=5, random_state=42, lengths=lengths
)
stability_3d

## Especificação de robustez (2D): só retorno + vol. realizada 21d

Versão mais parcimoniosa — `realized_vol_21d` e `realized_vol_63d` são
fortemente correlacionadas (Semana 3), então o modelo 3D pode estar
efetivamente perto de um 2D com peso redobrado na vol. de curto prazo.
Comparar os regimes das duas especificações é um teste de robustez barato.

In [ ]:
cols_2d = ["return_1d", "realized_vol_21d"]
obs_2d = portfolio[cols_2d].values

selection_2d = select_n_states(obs_2d, state_range=(2, 3, 4), n_init=20, random_state=42, lengths=lengths)
n_states_2d = selection_2d["bic"].idxmin()
model_2d = fit_gaussian_hmm(obs_2d, n_states=n_states_2d, n_init=30, random_state=42, lengths=lengths)

viterbi_2d = viterbi_decode(model_2d, obs_2d, lengths=lengths)
agreement_2d_vs_3d = (viterbi_2d == viterbi_states).mean() if n_states_2d == n_states_3d else None

print(f"2D seleciona {n_states_2d} estados (3D selecionou {n_states_3d})")
if agreement_2d_vs_3d is not None:
    print(f"Concordância Viterbi 2D vs. 3D: {agreement_2d_vs_3d:.1%}")
else:
    print("Nº de estados difere entre 2D e 3D — comparação direta de rótulo não é direta.")

characterize_regimes(model_2d, obs_2d, cols_2d)

## Salvar relatórios

In [ ]:
selection_3d.to_csv(TABLES_DIR / "hmm_state_selection_3d.csv")
regime_summary_3d.to_csv(TABLES_DIR / "hmm_regime_characterization_3d.csv")

pd.Series(stability_3d).drop("selected_n_states_distribution").to_csv(TABLES_DIR / "hmm_bootstrap_stability_3d.csv")
stability_3d["selected_n_states_distribution"].to_csv(TABLES_DIR / "hmm_bootstrap_n_states_distribution_3d.csv")

print("Relatórios salvos em", TABLES_DIR)
print("Regimes (Viterbi + Forward) salvos em", PROCESSED_DIR / "hmm_regimes_insample.parquet")